# EDA: Heart Disease Playground

Purpose: move through the requested plan Phases 1-4 via the shortest path.
Target data: `data/raw/train.csv`, `data/raw/test.csv`, `data/raw/sample_submission.csv`


In [1]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Infer project root (so it works even when run from nb/)
cwd = Path.cwd().resolve()
project_root = None
for p in [cwd] + list(cwd.parents):
    if (p / "data" / "raw" / "train.csv").exists():
        project_root = p
        break
if project_root is None:
    raise FileNotFoundError("project root not found: data/raw/train.csv")

DATA_DIR = project_root / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUB_PATH = DATA_DIR / "sample_submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sub = pd.read_csv(SUB_PATH)

print(train.shape, test.shape, sub.shape)


(630000, 15) (270000, 14) (270000, 2)


## 1. Data check (head)

In [2]:
display(train.head())
display(test.head())
display(sub.head())

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7
3,630003,44,0,3,112,177,0,0,168,0,0.9,1,0,3
4,630004,43,1,1,138,267,0,0,163,0,1.8,2,0,7


,id,Heart Disease
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0


## 2. Shapes, column diffs, dtype overview

In [3]:
# Shapes
print("train:", train.shape)
print("test:", test.shape)

# Column diffs
train_cols = set(train.columns)
test_cols = set(test.columns)
print("Only in train:", sorted(train_cols - test_cols))
print("Only in test:", sorted(test_cols - train_cols))

# dtypes
train.dtypes.to_frame("dtype").head(30)


train: (630000, 15)
test: (270000, 14)
Only in train: ['Heart Disease']
Only in test: []


,dtype
id,int64
Age,int64
Sex,int64
Chest pain type,int64
BP,int64
Cholesterol,int64
FBS over 120,int64
EKG results,int64
Max HR,int64
Exercise angina,int64


## 3. ID duplicates and train/test overlap

In [4]:
id_col = "id" if "id" in train.columns else None
if id_col:
    print("train dup id:", train[id_col].duplicated().sum())
    print("test dup id:", test[id_col].duplicated().sum())
    train_ids = set(train[id_col])
    test_ids = set(test[id_col])
    print("train/test overlap:", len(train_ids & test_ids))
else:
    print("No id column found.")


train dup id: 0
test dup id: 0
train/test overlap: 0


## 4. Missing rate / zero rate

In [5]:
def missing_zero_table(df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isna().mean().rename("missing_rate")
    zero_rate = (df == 0).mean(numeric_only=False).rename("zero_rate")
    uniq = df.nunique(dropna=True).rename("n_unique")
    dtype = df.dtypes.rename("dtype")
    out = pd.concat([dtype, uniq, missing, zero_rate], axis=1)
    return out.sort_values(["missing_rate", "zero_rate"], ascending=False)

mz = missing_zero_table(train)
mz.head(30)


,dtype,n_unique,missing_rate,zero_rate
FBS over 120,int64,2,0.0,0.920013
Exercise angina,int64,2,0.0,0.726275
Number of vessels fluro,int64,4,0.0,0.707717
EKG results,int64,3,0.0,0.508121
ST depression,float64,66,0.0,0.499903
Sex,int64,2,0.0,0.285265
id,int64,630000,0.0,0.000002
Age,int64,42,0.0,0.000000
Chest pain type,int64,4,0.0,0.000000
BP,int64,66,0.0,0.000000


## 5. Target distribution and label consistency

In [6]:
# Infer target column
target_col = None
for cand in ["Heart Disease", "HeartDisease", "target", "label"]:
    if cand in train.columns:
        target_col = cand
        break
print("target_col:", target_col)

if target_col:
    display(train[target_col].value_counts(dropna=False))
    display(train[target_col].value_counts(normalize=True, dropna=False))
    print("dtype:", train[target_col].dtype)


target_col: Heart Disease


Heart Disease
Absence     347546
Presence    282454
Name: count, dtype: int64

Heart Disease
Absence     0.55166
Presence    0.44834
Name: proportion, dtype: float64

dtype: object


## 6. Unique counts (categorical candidates)

In [7]:
# Numeric columns with few uniques are categorical candidates
n_unique = train.nunique(dropna=True).sort_values()
small_unique = n_unique[n_unique <= 20]
small_unique


Sex                        2
FBS over 120               2
Exercise angina            2
Heart Disease              2
EKG results                3
Slope of ST                3
Thallium                   3
Chest pain type            4
Number of vessels fluro    4
dtype: int64

## 7. Numeric summary (outlier candidates)

In [8]:
num_cols = train.select_dtypes(include=["number"]).columns
train[num_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T


,count,mean,std,min,1%,5%,50%,95%,99%,max
id,630000.0,314999.500000,181865.479132,0.0,6299.99,31499.95,314999.5,598499.05,623699.01,629999.0
Age,630000.0,54.136706,8.256301,29.0,35.00,41.00,54.0,67.00,71.00,77.0
Sex,630000.0,0.714735,0.451541,0.0,0.00,0.00,1.0,1.00,1.00,1.0
Chest pain type,630000.0,3.312752,0.851615,1.0,1.00,2.00,4.0,4.00,4.00,4.0
BP,630000.0,130.497433,14.975802,94.0,100.00,110.00,130.0,160.00,178.00,200.0
Cholesterol,630000.0,245.011814,33.681581,126.0,177.00,197.00,243.0,303.00,325.00,564.0
FBS over 120,630000.0,0.079987,0.271274,0.0,0.00,0.00,0.0,1.00,1.00,1.0
EKG results,630000.0,0.981660,0.998783,0.0,0.00,0.00,0.0,2.00,2.00,2.0
Max HR,630000.0,152.816763,19.112927,71.0,103.00,114.00,157.0,179.00,182.00,202.0
Exercise angina,630000.0,0.273725,0.445870,0.0,0.00,0.00,0.0,1.00,1.00,1.0


## 8. Relationship to target

- Numeric: mean/median by target
- Categorical: target rate by category


In [9]:
if target_col:
    # Numeric
    num_cols_no_target = [c for c in num_cols if c != target_col]
    if len(num_cols_no_target) > 0:
        display(train.groupby(target_col)[num_cols_no_target].agg(["mean", "median"]).T)

    # Categorical candidates
    cat_candidates = [c for c in train.columns if train[c].nunique(dropna=True) <= 20 and c != target_col]

    def target_rate_by_category(col: str) -> pd.DataFrame:
        return (
            train.groupby(col)[target_col]
            .value_counts(normalize=True)
            .rename("rate")
            .reset_index()
        )

    for c in cat_candidates:
        display(target_rate_by_category(c).head(50))


Heart Disease                         Absence       Presence
id                      mean    314965.283965  315041.601178
                        median  314835.500000  315206.000000
Age                     mean        52.558093      56.079114
                        median      52.000000      57.000000
Sex                     mean         0.575337       0.886258
                        median       1.000000       1.000000
Chest pain type         mean         2.959070       3.747941
                        median       3.000000       4.000000
BP                      mean       130.567381     130.411366
                        median     130.000000     130.000000
Cholesterol             mean       242.499102     248.103585
                        median     239.000000     246.000000
FBS over 120            mean         0.071778       0.090089
                        median       0.000000       0.000000
EKG results             mean         0.784506       1.224249
                        median       0.000000       2.000000
Max HR                  mean       160.415105     143.467372
                        median     162.000000     146.000000
Exercise angina         mean         0.096117       0.492264
                        median       0.000000       0.000000
ST depression           mean         0.347808       1.169104
                        median       0.000000       1.200000
Slope of ST             mean         1.251877       1.706876
                        median       1.000000       2.000000
Number of vessels fluro mean         0.135291       0.839553
                        median       0.000000       1.000000
Thallium                mean         3.553955       5.929203
                        median       3.000000       7.000000

,Sex,Heart Disease,rate
0,0,Absence,0.821236
1,0,Presence,0.178764
2,1,Presence,0.555933
3,1,Absence,0.444067


,Chest pain type,Heart Disease,rate
0,1,Absence,0.891931
1,1,Presence,0.108069
2,2,Absence,0.837819
3,2,Presence,0.162181
4,3,Absence,0.809335
5,3,Presence,0.190665
6,4,Presence,0.697478
7,4,Absence,0.302522


,FBS over 120,Heart Disease,rate
0,0,Absence,0.556583
1,0,Presence,0.443417
2,1,Presence,0.504961
3,1,Absence,0.495039


,EKG results,Heart Disease,rate
0,0,Absence,0.658502
1,0,Presence,0.341498
2,1,Absence,0.639939
3,1,Presence,0.360061
4,2,Presence,0.559560
5,2,Absence,0.440440


,Exercise angina,Heart Disease,rate
0,0,Absence,0.686567
1,0,Presence,0.313433
2,1,Presence,0.806288
3,1,Absence,0.193712


,Slope of ST,Heart Disease,rate
0,1,Absence,0.737743
1,1,Presence,0.262257
2,2,Presence,0.692067
3,2,Absence,0.307933
4,3,Presence,0.721082
5,3,Absence,0.278918


,Number of vessels fluro,Heart Disease,rate
0,0,Absence,0.696868
1,0,Presence,0.303132
2,1,Presence,0.729346
3,1,Absence,0.270654
4,2,Presence,0.897078
5,2,Absence,0.102922
6,3,Presence,0.899549
7,3,Absence,0.100451


,Thallium,Heart Disease,rate
0,3,Absence,0.801951
1,3,Presence,0.198049
2,6,Presence,0.686394
3,6,Absence,0.313606
4,7,Presence,0.815391
5,7,Absence,0.184609


## 9. Train vs Test distribution drift (simple)

- Continuous: KS statistic / PSI (simple)
- Categorical: rate differences


In [10]:
from scipy.stats import ks_2samp

# PSI (simple)
def psi(expected, actual, buckets=10):
    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()
    if expected.nunique() <= 1 or actual.nunique() <= 1:
        return 0.0
    quantiles = np.linspace(0, 1, buckets + 1)
    bins = np.unique(np.quantile(expected, quantiles))
    if len(bins) <= 2:
        return 0.0
    expected_counts = pd.cut(expected, bins=bins, include_lowest=True).value_counts(normalize=True)
    actual_counts = pd.cut(actual, bins=bins, include_lowest=True).value_counts(normalize=True)
    # align
    expected_counts, actual_counts = expected_counts.align(actual_counts, fill_value=1e-6)
    psi_val = np.sum((expected_counts - actual_counts) * np.log(expected_counts / actual_counts))
    return float(psi_val)

num_cols_train = [c for c in num_cols if c in test.columns]

rows = []
for c in num_cols_train:
    ks_stat = ks_2samp(train[c].dropna(), test[c].dropna()).statistic
    psi_val = psi(train[c], test[c])
    rows.append((c, ks_stat, psi_val))

drift_num = pd.DataFrame(rows, columns=["col", "ks", "psi"]).sort_values(["psi", "ks"], ascending=False)
drift_num.head(20)


,col,ks,psi
0,id,1.000000,1.151281e+01
10,ST depression,0.002571,4.616251e-05
1,Age,0.002162,4.082804e-05
8,Max HR,0.001737,3.804858e-05
4,BP,0.002328,3.716932e-05
3,Chest pain type,0.002090,1.986623e-05
5,Cholesterol,0.001396,1.829930e-05
12,Number of vessels fluro,0.002210,1.182539e-05
11,Slope of ST,0.003089,6.445036e-06
13,Thallium,0.000311,4.047943e-07


### Categorical rate differences

In [11]:
cat_cols = [c for c in train.columns if c in test.columns and train[c].nunique(dropna=True) <= 20]

rows = []
for c in cat_cols:
    train_rate = train[c].value_counts(normalize=True)
    test_rate = test[c].value_counts(normalize=True)
    diff = (train_rate - test_rate).abs().fillna(0).sum() / 2
    rows.append((c, diff))

drift_cat = pd.DataFrame(rows, columns=["col", "total_variation_distance"]).sort_values("total_variation_distance", ascending=False)
drift_cat.head(20)


,col,total_variation_distance
5,Slope of ST,0.003089
6,Number of vessels fluro,0.002210
1,Chest pain type,0.002090
3,EKG results,0.001618
0,Sex,0.001580
4,Exercise angina,0.000930
7,Thallium,0.000311
2,FBS over 120,0.000065
